Import and configure

In [1]:
import os
import requests
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import json

PROJECT_ROOT = "D:/Crop_yeild_system"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")  
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

crop = "Rabi crop"
country = "India"


    NASA POWER WEATHER DATA 

In [2]:
import requests
import pandas as pd
import os

print("[INFO] NASA POWER weather data loading......")

# Define coordinates for 5 states (approximate center of the state)
states_coords = {
    "Chandigarh": (30.7333, 76.7794),
    "Punjab": (31.1471, 75.3412),
    "Haryana": (29.0588, 76.0856),
    "Rajasthan": (27.0238, 74.2179),
    "UttarPradesh": (26.8467, 80.9462)
}

def fetch_nasa_weather(lat, lon, start_date, end_date):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start": start_date,
        "end": end_date,
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M",
        "community": "AG",
        "format": "JSON"
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    df = pd.DataFrame(data["properties"]["parameter"])
    df.index = pd.to_datetime(df.index)
    return df

def add_season_column(df):
    """
    Adds a 'Crop_Season' column to the daily data:
    - Rabi: Oct (prev year) – Apr (current year)
    - Kharif: Jun – Sep
    """
    df = df.copy()
    df['Month'] = df.index.month
    df['Year'] = df.index.year
    df['Crop_Season'] = None
    
    # Assign Rabi
    df.loc[((df['Month'] >= 10) | (df['Month'] <= 4)), 'Crop_Season'] = 'Rabi'
    # Assign Kharif
    df.loc[df['Month'].between(6, 9), 'Crop_Season'] = 'Kharif'
    
    return df

def compute_seasonal_averages(df):
    """
    Computes seasonal averages from daily data
    """
    seasonal_df = df.groupby(['Year', 'Crop_Season']).agg({
        'T2M':'mean',
        'T2M_MAX':'mean',
        'T2M_MIN':'mean',
        'PRECTOTCORR':'sum',
        'ALLSKY_SFC_SW_DWN':'mean',
        'RH2M':'mean',
        'WS2M':'mean'
    }).reset_index()
    return seasonal_df

# -----------------------------
# Fetch daily data for all states
# -----------------------------
start_date = "20110101"
end_date   = "20221231"

all_states_daily = []

for state, (lat, lon) in states_coords.items():
    print(f"[INFO] Fetching data for {state}...")
    try:
        df_state = fetch_nasa_weather(lat, lon, start_date, end_date)
        df_state = add_season_column(df_state)
        df_state['State'] = state
        all_states_daily.append(df_state)
        print(f"[INFO] Daily data loaded for {state}")
    except Exception as e:
        print(f"[ERROR] Failed for {state}: {e}")

# Combine all states daily data
final_daily_df = pd.concat(all_states_daily)
final_daily_df.reset_index(inplace=True)
final_daily_df.rename(columns={'index':'Date'}, inplace=True)

# Save daily data
daily_csv_path = os.path.join(RAW_DIR, "nasa_power_weather_daily.csv")
final_daily_df.to_csv(daily_csv_path, index=False)
print(f"[INFO] All daily weather data saved at {daily_csv_path}")

# -----------------------------
# Compute seasonal averages
# -----------------------------
all_states_seasonal = []

for state in final_daily_df['State'].unique():
    df_state = final_daily_df[final_daily_df['State']==state]
    seasonal_df = compute_seasonal_averages(df_state)
    seasonal_df['State'] = state
    all_states_seasonal.append(seasonal_df)

final_seasonal_df = pd.concat(all_states_seasonal, ignore_index=True)

# Save seasonal data
seasonal_csv_path = os.path.join(RAW_DIR, "nasa_power_weather_seasonal.csv")
final_seasonal_df.to_csv(seasonal_csv_path, index=False)
print(f"[INFO] All seasonal weather data saved at {seasonal_csv_path}")


[INFO] NASA POWER weather data loading......
[INFO] Fetching data for Chandigarh...
[INFO] Daily data loaded for Chandigarh
[INFO] Fetching data for Punjab...
[INFO] Daily data loaded for Punjab
[INFO] Fetching data for Haryana...
[INFO] Daily data loaded for Haryana
[INFO] Fetching data for Rajasthan...
[INFO] Daily data loaded for Rajasthan
[INFO] Fetching data for UttarPradesh...
[INFO] Daily data loaded for UttarPradesh
[INFO] All daily weather data saved at D:/Crop_yeild_system\data\raw\nasa_power_weather_daily.csv
[INFO] All seasonal weather data saved at D:/Crop_yeild_system\data\raw\nasa_power_weather_seasonal.csv


NDVI EXTRACTION USING GEE

In [3]:
import ee
import pandas as pd
import os
# Authenticate & Initialize
ee.Initialize(project="ee-alfeenubaidh2006")

# 1. Load India state boundaries and filter states
states = ee.FeatureCollection("FAO/GAUL/2015/level1") \
            .filter(ee.Filter.eq("ADM0_NAME", "India"))

state_names = ['Chandigarh', 'Punjab', 'Haryana', 'Rajasthan', 'Uttar Pradesh']
roi = states.filter(ee.Filter.inList("ADM1_NAME", state_names))

# 2. Load MODIS NDVI
modis = (ee.ImageCollection('MODIS/006/MOD13A1')
          .select('NDVI')
          .filterDate('2011-01-01', '2022-12-31')
          .map(lambda img: img.multiply(0.0001)
               .copyProperties(img, ['system:time_start'])))

# 3. Build year-month combinations
years = list(range(2011, 2023))
months = list(range(1, 13))

# 4. Compute monthly means
def make_monthly(y, m):
    monthly = (modis.filter(ee.Filter.calendarRange(y, y, 'year'))
                     .filter(ee.Filter.calendarRange(m, m, 'month')))
    mean = monthly.mean().set({'year': y, 'month': m})
    return mean

monthly_images = [make_monthly(y, m) for y in years for m in months]
monthly_ndvi = ee.ImageCollection.fromImages(monthly_images)

# 5. Reduce over states
def zonal_stats(img):
    reduced = img.reduceRegions(
        collection=roi,
        reducer=ee.Reducer.mean(),
        scale=500
    )
    return reduced.map(lambda f: f.set({'year': img.get('year'), 'month': img.get('month')}))

all_stats = monthly_ndvi.map(zonal_stats).flatten()

# 6. Convert to Pandas
# (Get features as list → client-side → pandas)
features = all_stats.aggregate_array('.geo').getInfo()  # geometry not needed, just to trigger
data = all_stats.aggregate_array('mean').getInfo()      # NDVI values
years_out = all_stats.aggregate_array('year').getInfo()
months_out = all_stats.aggregate_array('month').getInfo()
states_out = all_stats.aggregate_array('ADM1_NAME').getInfo()

df = pd.DataFrame({
    "State": states_out,
    "Year": years_out,
    "Month": months_out,
    "NDVI": data
})

# 7. Save to CSV
csv_path = os.path.join(RAW_DIR, "NDVI_states_2011_2022.csv")
df.to_csv(csv_path, index=False)
print("Exported NDVI data to CSV ✅")


c:\Users\ASUS\anaconda3\envs\cropredcition_exe\lib\site-packages\ee\deprecation.py:209: DeprecationWarning: 

Attention required for MODIS/006/MOD13A1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD13A1

  warnings.warn(warning, category=DeprecationWarning)


Exported NDVI data to CSV ✅


In [ ]:
import geopandas as gpd
import pandas as pd
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import box
import numpy as np
import os
# Load shapefile
shapefile = "C:\\Users\\ASUS\\Downloads\\gadm41_IND_shp\\gadm41_IND_1.shp"
gdf = gpd.read_file(shapefile)

# Filter states
states_of_interest = ["Chandigarh", "Punjab", "Haryana", "Rajasthan", "Uttar Pradesh"]
gdf = gdf[gdf["NAME_1"].isin(states_of_interest)]

# Load raster
raster_file = "C:\\Users\\ASUS\\Downloads\\organic_carbon_0cm_250m.tif"
raster = rasterio.open(raster_file)

# Function to create grid cells inside a polygon
def create_grid(polygon, cell_size):
    minx, miny, maxx, maxy = polygon.bounds
    x_coords = np.arange(minx, maxx, cell_size)
    y_coords = np.arange(miny, maxy, cell_size)
    grid_cells = []
    for x in x_coords:
        for y in y_coords:
            cell = box(x, y, x+cell_size, y+cell_size)
            if polygon.intersects(cell):
                grid_cells.append(cell.intersection(polygon))
    return grid_cells

# Create a new GeoDataFrame for grid cells
all_cells = []
cell_size = 0.1  # ~0.1 degree (~10 km, adjust as needed)
for row in gdf.itertuples():
    cells = create_grid(row.geometry, cell_size)
    for cell in cells:
        all_cells.append({"State": row.NAME_1, "geometry": cell})

grid_gdf = gpd.GeoDataFrame(all_cells, crs=gdf.crs)

# Compute zonal stats for each cell
stats = zonal_stats(grid_gdf, raster_file, stats=["mean", "min", "max", "median", "std"], nodata=None)
records = []
for feature, row in zip(stats, grid_gdf.itertuples()):
    record = {
        "State": row.State,
        "Mean_SOC": feature["mean"],
        "Median_SOC": feature["median"],
        "Min_SOC": feature["min"],
        "Max_SOC": feature["max"],
        "Std_SOC": feature["std"]
    }
    records.append(record)

df = pd.DataFrame(records)

# Save to CSV
csv_path = os.path.join(RAW_DIR,"india_states_soc_stats.csv")
df.to_csv(csv_path, index=False)

print("Saved zonal stats to india_states_soc_stats.csv")



Saved zonal stats to india_states_soc_stats.csv
